In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from IPython.display import clear_output
from matplotlib import cm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C
from scipy.optimize import minimize

In [2]:
# Load all data
folder = Path(r'C:\Users\Dave Amin\Documents\docs-Sync\Career\2025-10 IC AI course\Modules\Capstone2\initial_data')
folder2 = Path(r'function_3')

def load_data():
    file_path_inputs =  folder / folder2 / rf"initial_inputs.npy"
    file_path_outputs = folder / folder2 / rf"initial_outputs.npy"
    X = np.load(file_path_inputs)
    Y = np.load(file_path_outputs)
    print('Initial data :','X.shape', X.shape, 'Y.shape', Y.shape)
    
    X_new_point = np.array([9.40184e-01, 6.55584e-01, 7.00000e-06], dtype=np.float64)
    Y_new_point = np.array([-0.10681450997226602], dtype=np.float64)
    X = np.vstack((X, X_new_point))
    Y = np.append(Y, Y_new_point)
    print('Added week 1 :','X.shape', X.shape, 'Y.shape', Y.shape)

    # week 2 new values go here !!!!
  
    return X,Y

X,Y = load_data()

Initial data : X.shape (15, 3) Y.shape (15,)
Added week 1 : X.shape (16, 3) Y.shape (16,)


In [3]:
def printMinMaxDiff(label,Q,N=6):
    print(label,"; min=", f"{np.min(Q):.{N}g}" ,
                "; max=", f"{np.max(Q):.{N}g}" ,
                "; diff=", f"{np.max(Q)-np.min(Q):.{N}g}"  )
# printMinMaxDiff("X col1",X[:, 0])
# printMinMaxDiff("X col2",X[:, 1])
# printMinMaxDiff("Y col1",Y)

In [4]:
print("X",X)
print("Y",Y)

X [[1.71525207e-01 3.43916870e-01 2.48737201e-01]
 [2.42114461e-01 6.44074270e-01 2.72432809e-01]
 [5.34905720e-01 3.98500915e-01 1.73388729e-01]
 [4.92581415e-01 6.11593188e-01 3.40176386e-01]
 [1.34621666e-01 2.19917240e-01 4.58206220e-01]
 [3.45523271e-01 9.41359831e-01 2.69363479e-01]
 [1.51836632e-01 4.39990619e-01 9.90881867e-01]
 [6.45502835e-01 3.97142940e-01 9.19771338e-01]
 [7.46911945e-01 2.84196309e-01 2.26299855e-01]
 [1.70476994e-01 6.97032401e-01 1.49169434e-01]
 [2.20549337e-01 2.97825244e-01 3.43555344e-01]
 [6.66013659e-01 6.71985151e-01 2.46295297e-01]
 [4.68089497e-02 2.31360241e-01 7.70617592e-01]
 [6.00097282e-01 7.25135725e-01 6.60886415e-02]
 [9.65994849e-01 8.61119690e-01 5.66829131e-01]
 [9.40184000e-01 6.55584000e-01 7.00000000e-06]]
y [-0.1121222  -0.08796286 -0.11141465 -0.03483531 -0.04800758 -0.11062091
 -0.39892551 -0.11386851 -0.13146061 -0.09418956 -0.04694741 -0.10596504
 -0.11804826 -0.03637783 -0.05675837 -0.10681451]


## Matern Kernel

In [11]:
Matern_kernel = C(1.0, (1e-3, 1e3)) * Matern(length_scale=[0.5] * X.shape[1], 
                          length_scale_bounds=(1e-2, 2.0),      # Cap at 2.0 to force local relevance
                          nu=2.5         
                          ) + WhiteKernel(noise_level=1e-3, 
                          noise_level_bounds=(1e-5, 0.1))

model = GaussianProcessRegressor(
    kernel=Matern_kernel,
    # alpha=0,               # Let WhiteKernel handle the noise instead
    normalize_y=True,        # CRITICAL: This scales those tiny e-124 values so the GP can see them
    n_restarts_optimizer=30)   # Gives the optimizer N tries to find the best fit
# 3. Fit the model
model.fit(X, Y)
print(f"Optimized Kernel: {model.kernel_}")

Optimized Kernel: 1.14**2 * Matern(length_scale=[2, 2, 0.0628], nu=2.5) + WhiteKernel(noise_level=0.01)


C:\Anaconda\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Anaconda\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [13]:
from scipy.optimize import minimize
from scipy.stats import norm

def expected_improvement(x, model, best_y):
    mu, sigma = model.predict(x.reshape(1, -1), return_std=True)
    if sigma <= 0: return 0
    # We want to exceed our current best (maximized negative side effects)
    z = (mu - best_y) / sigma
    ei = (mu - best_y) * norm.cdf(z) + sigma * norm.pdf(z)
    return ei[0]

# 3. GLOBAL SEARCH + LOCAL POLISH
current_best_y = np.max(Y)
num_candidates = 50000
x_candidates = np.random.uniform(0, 1, size=(num_candidates, n_dims))

# Evaluate EI on candidates
ei_values = np.array([expected_improvement(c, model, current_best_y) for c in x_candidates])
x_start = x_candidates[np.argmax(ei_values)]

# Polisher
res = minimize(
    lambda x: -expected_improvement(x, model, current_best_y),
    x0=x_start, 
    bounds=[(0, 1)] * n_dims, 
    method='L-BFGS-B'
)

formatted_submission_string = "-".join(f"{val:.6f}" for val in res.x)
print(f"Submit this: {formatted_submission_string}")

y_predicted = model.predict(res.x.reshape(1, -1))[0]
print(f"Predicted Y is {model.predict(res.x.reshape(1,-1))[0]:.4f} ")


Submit this: 1.000000-1.000000-0.393312
Predicted Y is -0.0493 


In [103]:
num = 1

for i in range(0, num):
    print("count")

count
